# v3 vs Gemini on messy logs

20 lines in seven formats that are **not** LogHub. `rule_parser.py` returns
`None` on 19 of 20, so the parser is out of the picture and this is a straight
head-to-head.

No gold labels. Nothing here prints an accuracy number — the last cell marks
where the two arms disagree, and you decide who is right against
`schema_v2.SPEC_EVAL`.

**Runtime → Change runtime type → T4 GPU.** Five cells, ~8 min, ~$0.02.

In [ ]:
# 1. setup (~3 min) -- restarts once, then re-run this same cell
import os
os.chdir('/content')
!rm -rf tiny-log-parser
!git clone -q https://github.com/arshirazi97/tiny-log-parser.git
os.chdir('/content/tiny-log-parser')
!pip install -q "transformers==4.51.3" "peft==0.20.0" accelerate bitsandbytes openai
!pip uninstall -q -y torchao      # peft 0.20 raises on torchao < 0.16; Colab ships 0.10

import importlib.util, transformers, peft
stale = ((transformers.__version__, peft.__version__) != ("4.51.3", "0.20.0")
         or importlib.util.find_spec("torchao") is not None)
if stale:
    print("restarting to pick up the new versions...")
    os.kill(os.getpid(), 9)

In [ ]:
# 2. build the corpus, and show the parser has no coverage here
import os
os.chdir('/content/tiny-log-parser')
!python real-eval/messy_corpus.py messy.log -o real-eval/corpus_messy.jsonl
!python real-eval/predict.py --arm rules --corpus real-eval/corpus_messy.jsonl --out real-eval/preds_messy_rules.jsonl

Expect `20 rows, 19 unparseable` above. That is the point of this corpus.

To use your own lines instead, write them to a file and rerun the cell above
against it — one log line per line, no labels needed:

```python
open('mine.log','w').write('''<paste your log lines here>''')
!python real-eval/messy_corpus.py mine.log -o real-eval/corpus_messy.jsonl
```

In [ ]:
# 3. v3 (~2 min, mostly model load)
!python real-eval/predict.py --arm model --adapter arshirazi/tiny-log-parser-v3 --corpus real-eval/corpus_messy.jsonl --batch 32 --out real-eval/preds_messy_v3.jsonl

In [ ]:
# 4. Gemini (~1 min, ~$0.02). --shots 0 so both arms get the identical prompt.
import getpass, os
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ').strip()

!python real-eval/predict.py --arm gemini --shots 0 --corpus real-eval/corpus_messy.jsonl --out real-eval/preds_messy_gemini.jsonl

In [ ]:
# 5. the comparison
!python real-eval/compare_arms.py --corpus real-eval/corpus_messy.jsonl v3=real-eval/preds_messy_v3.jsonl gemini=real-eval/preds_messy_gemini.jsonl

Add `--only-disagreements` to the cell above to skip the lines both arms agree
on. Agreement is not accuracy — both can be wrong on the same line.

Four things to judge by hand:

- **`level` on lines that carry none.** `Error -60005 creating authorization`
  has no level field; `ERROR` there is prose. v3 held 0/32 on gold-null in the
  labelled eval — check it survives on formats it has never seen.
- **`latency_ms` vs a duration in prose.** `took=4.775s` and `rt=7.881` are
  structural; `finished in 6.049 s` is not. This is the family v3 was trained on.
- **`service` on the truncated line** (`<134>Jul 12 18:44:0`). No service token
  exists. Anything non-null is invention.
- **`timestamp` with no year.** The schema says a `1900` sentinel, not a guess —
  and several lines here do carry a real year, so both behaviours should appear.